| Feature                | FAISS   | ChromaDB |
| ---------------------- | ------- | -------- |
| Flat index             | ✅       | ❌        |
| IVF                    | ✅       | ❌        |
| HNSW                   | ✅       | ✅        |
| PQ                     | ✅       | ❌        |
| Manual index selection | ✅       | ❌        |
| Metadata filtering     | Limited | ✅        |
| Persistence            | Manual  | Built-in |
| LangChain integration  | ✅       | ✅        |


Final understanding
FAISS = vector index/search engine

You manually manage:
- index type
- dimension
- metric
- docstore
- ID mapping
- persistence
  
Chroma = vector database
It manages:
- vectors
- documents
- metadata
- IDs
- collections
- index
- persistence


FAISS is a low-level, high-performance library for dense-vector similarity search and clustering. It gives developers direct control over index structures such as Flat, IVF, HNSW and product-quantized indexes, and it offers strong CPU and GPU capabilities. However, FAISS is not a complete vector database: document storage, metadata management, filtering, CRUD APIs, collections, persistence orchestration and server infrastructure generally need to be handled separately.

Chroma is a retrieval database/search infrastructure designed for AI applications. It stores embeddings together with documents, metadata and IDs, and provides collections, persistence, metadata filtering, full-text and sparse retrieval, CRUD operations and client-server or hosted deployment options. In current Chroma, single-node vector search uses HNSW, while its broader schema and cloud architecture also support other retrieval indexes such as SPANN and sparse/full-text indexes.

Therefore, FAISS is preferable when low-level index control, custom ANN algorithms, compression or GPU optimization is the priority. Chroma is preferable when building a complete RAG application that needs database-style storage, filtering, updates and operational simplicity.

| Feature         | FAISS                 | Chroma |
| --------------- | --------------------- | ------ |
| Store vectors   | ✅                     | ✅      |
| Store documents | ❌                     | ✅      |
| Store metadata  | ❌                     | ✅      |
| Collections     | ❌                     | ✅      |
| CRUD            | Limited               | ✅      |
| Filtering       | ❌ (native)            | ✅      |
| Persistence     | Basic index save/load | ✅      |
| Client APIs     | ❌                     | ✅      |
| Server mode     | ❌                     | ✅      |


| Component        | LangChain + FAISS             | Chroma                          |
| ---------------- | ----------------------------- | ------------------------------- |
| Embeddings       | Native FAISS index            | Chroma vector index             |
| Documents        | LangChain `Docstore`          | Chroma collection               |
| Metadata         | LangChain `Document.metadata` | Chroma collection record        |
| Mapping          | `index_to_docstore_id`        | Internally managed              |
| Save             | FAISS file + pickle           | Database persistence            |
| Metadata filters | Wrapper/application handling  | Native database filtering       |
| Collections      | Not native to FAISS           | Native                          |
| CRUD             | Wrapper/index-dependent       | Native record operations        |
| Server/cloud     | Separate system required      | Supported database architecture |


## Import Libraries

In [2]:
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
load_dotenv()

True

## Configurations

In [3]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [4]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
model = ChatOpenAI(model="gpt-4o-mini")

## Load PDF

In [5]:
file_path = r"data\\llama2-research-paper.pdf"
loader = PyPDFLoader(file_path)
pages = loader.load()
print("Total pages:", len(pages))

Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.


Total pages: 77


## Create Chunks

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

chunks = text_splitter.split_documents(pages)
print("Total chunks:", len(chunks))

Total chunks: 175


## Create Vector Store

In [7]:
vector_store = Chroma(
    collection_name="llama2_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_db_llama2",
    collection_metadata={
        "hnsw:space": "cosine"
    }
)

In [8]:
{
    "id": "doc-101",
    "embedding": [0.12, -0.45, 0.78, ...],
    "document": "Employees receive 20 days of annual leave.",
    "metadata": {
        "source": "hr_policy.pdf",
        "page": 5,
        "department": "HR"
    }
}

{'id': 'doc-101',
 'embedding': [0.12, -0.45, 0.78, Ellipsis],
 'document': 'Employees receive 20 days of annual leave.',
 'metadata': {'source': 'hr_policy.pdf', 'page': 5, 'department': 'HR'}}

## Add Documents

In [9]:
document_ids = vector_store.add_documents(
    documents=chunks
)

print("Documents added:", len(document_ids))

print(
    "Total documents stored:",
    vector_store._collection.count()
)

Documents added: 175
Total documents stored: 350


## Create Retriever

In [10]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    }
)

In [11]:
query = "What is the architecture of Llama 2?"

retrieved_documents = retriever.invoke(query)

for i, document in enumerate(
    retrieved_documents,
    start=1
):
    print(f"\n--- Retrieved document {i} ---")
    print(document.page_content[:500])
    print("Metadata:", document.metadata)


--- Retrieved document 1 ---
capabilities and limitations of our models; results can be found in Section 4.1.
2.2 Training Details
We adopt most of the pretraining setting and model architecture fromLlama 1. We use the standard
transformer architecture (Vaswani et al., 2017), apply pre-normalization using RMSNorm (Zhang and
Sennrich, 2019), use the SwiGLU activation function (Shazeer, 2020), and rotary positional embeddings
(RoPE, Su et al. 2022). The primary architectural differences fromLlama 1 include increased context l
Metadata: {'page': 4, 'creationdate': '2023-07-20T00:30:36+00:00', 'trapped': '/False', 'source': 'data\\\\llama2-research-paper.pdf', 'keywords': '', 'subject': '', 'producer': 'pdfTeX-1.40.25', 'total_pages': 77, 'title': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'creator': 'LaTeX with hyperref', 'moddate': '2023-07-20T00:30:36+00:00', 'page_label': '5', 'author': ''}

--- Retrieved document 2

## Prompt Template

In [12]:
prompt = ChatPromptTemplate.from_template(
    """
    You are a question-answering assistant.

    Answer the question only from the provided context.

    If the context does not contain the answer, say:
    "I do not have enough information in the provided document."

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
)

In [13]:
def format_docs(docs):
    return "\n\n".join(
        f"""
        Source: {doc.metadata.get("source")}
        Page: {doc.metadata.get("page")}

        {doc.page_content}
        """
        for doc in docs
    )

## Creating RAG Chain

In [14]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

## Asking Question

In [15]:
answer = rag_chain.invoke(
    "What is the architecture of Llama 2?"
)

print("\nFinal answer:\n")
print(answer)


Final answer:

The architecture of Llama 2 is based on the standard transformer architecture, applies pre-normalization using RMSNorm, uses the SwiGLU activation function, and incorporates rotary positional embeddings (RoPE). Key architectural differences from Llama 1 include increased context length and grouped-query attention (GQA).


## Loading Vector Store

In [16]:
loaded_vector_store = Chroma(
    collection_name="llama2_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_db_llama2"
)

In [17]:
loaded_retriever = loaded_vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [18]:
docs = loaded_retriever.invoke(
    "What is Llama 2?"
)

for doc in docs:
    print(doc.page_content[:500])

Figure 4: Training ofLlama 2-Chat: This process begins with thepretraining of Llama 2 using publicly
available online sources. Following this, we create an initial version ofLlama 2-Chatthrough the application
of supervised fine-tuning. Subsequently, the model is iteratively refined using Reinforcement Learning
with Human Feedback(RLHF) methodologies, specifically through rejection sampling and Proximal Policy
Optimization (PPO). Throughout the RLHF stage, the accumulation ofiterative reward mod
Figure 4: Training ofLlama 2-Chat: This process begins with thepretraining of Llama 2 using publicly
available online sources. Following this, we create an initial version ofLlama 2-Chatthrough the application
of supervised fine-tuning. Subsequently, the model is iteratively refined using Reinforcement Learning
with Human Feedback(RLHF) methodologies, specifically through rejection sampling and Proximal Policy
Optimization (PPO). Throughout the RLHF stage, the accumulation ofiterative reward mo